# 메시지 타입 — Human / AI / Tool / System

LLM 과의 대화는 한 덩어리 텍스트가 아니라 **"역할(role)" 이 붙은 메시지들의 리스트**다. LangGraph 에이전트는 이 메시지 타입을 실제로 **구분해서 생성·판단·조립**한다.

| 타입 | role | 누가/무엇이 만드나 | 의미 |
|---|---|---|---|
| `SystemMessage` | system | 개발자 | 페르소나·지시 (예: "너는 영어 교정 도우미") |
| `HumanMessage` | human | 사용자 | 사용자 입력 |
| `AIMessage` | ai | LLM | LLM 응답 (+ 도구 호출 의도 `tool_calls`) |
| `ToolMessage` | tool | 도구 실행 결과 | 도구가 돌려준 값 |

`AnyMessage` 는 "이 넷 중 무엇이든" 을 뜻하는 **타입 힌트용 유니온** 이다. 그래서 [basics 복습] State 정의에서 `list[AnyMessage]` 로 쓴다.

> 이 노트북은 LLM/API 키 없이 돌아간다 (메시지 객체와 ToolNode 동작만 본다).

## 1. 메시지 타입별 정체 확인

각 메시지는 `.type`(role)과 `.content`(내용)를 가진다. 타입에 따라 추가 필드가 다르다 — `AIMessage` 는 `.tool_calls`, `ToolMessage` 는 `.tool_call_id` 를 가진다.

In [ ]:
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, ToolMessage,
)

msgs = [
    SystemMessage("너는 친절한 도우미야."),
    HumanMessage("안녕?"),
    AIMessage("안녕하세요! 무엇을 도와드릴까요?"),
    ToolMessage(content="42", tool_call_id="call_1"),
]

for m in msgs:
    print(f"{type(m).__name__:14s} | role={m.type:7s} | content={m.content!r}")

## 2. 노드에서 메시지 타입을 *생성*한다

에이전트의 각 노드는 자기 역할에 맞는 메시지 타입을 만든다.
- `chatbot`/`agent` 노드 → LLM 이 `AIMessage` 생성 (도구가 필요하면 `.tool_calls` 포함)
- `tools` 노드(`ToolNode`) → 도구 실행 후 `ToolMessage` 생성

LLM 없이, 도구 호출 의도가 담긴 `AIMessage` 를 직접 만들어 `ToolNode` 에 흘려보내며 **어떤 타입의 메시지가 쌓이는지** 관찰한다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더한다."""
    return a + b

# [basics 복습] 메시지를 누적하는 State (Annotated + add_messages 리듀서)
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# [basics 복습] ToolNode 를 노드로 등록한 미니 그래프
g = StateGraph(State)
g.add_node("tools", ToolNode([add]))
g.add_edge(START, "tools")
g.add_edge("tools", END)
graph = g.compile()

# LLM 이 만들었을 법한 'AIMessage + tool_calls' 를 직접 구성
ai = AIMessage(
    content="",
    tool_calls=[{"name": "add", "args": {"a": 2, "b": 5}, "id": "call_1", "type": "tool_call"}],
)

out = graph.invoke({"messages": [HumanMessage("2 더하기 5는?"), ai]})

for m in out["messages"]:
    extra = ""
    if isinstance(m, AIMessage):
        extra = f"  tool_calls={m.tool_calls}"
    elif isinstance(m, ToolMessage):
        extra = f"  tool_call_id={m.tool_call_id}"
    print(f"{type(m).__name__:13s} | content={m.content!r}{extra}")

위 출력의 메시지 누적 순서가 곧 **에이전트 한 턴의 흐름**이다:

```
HumanMessage  : 사용자 질문
AIMessage     : LLM 이 '도구를 부르겠다'는 의도(tool_calls) 생성
ToolMessage   : ToolNode 가 도구를 실행한 결과 (여기선 add → 7)
```

`ToolMessage` 의 `tool_call_id` 는 어떤 `AIMessage` 의 도구 호출에 대한 결과인지 짝지어주는 연결고리다.

## 3. 노드/라우터에서 메시지 타입으로 *분기*한다

메시지 타입(또는 위치)을 보고 다음 동작을 정하는 게 그래프 제어의 핵심이다. 대표 패턴 둘:

1. **마지막 메시지가 `AIMessage` 이고 `tool_calls` 가 있으면 → 도구 실행 노드로** (조건부 라우팅)
2. **`ToolMessage`(도구 결과)를 꺼내 후처리** (예: 구조화 응답 생성)

In [ ]:
def route(messages):
    """마지막 메시지를 보고 다음 행선지를 정하는 라우터 (개념 시연)"""
    last = messages[-1]
    # [basics 복습] add_conditional_edges 의 라우터가 하는 판단과 동일한 로직
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"     # 도구 호출 의도가 있으면 도구 노드로
    return "END"           # 없으면 종료

# 케이스 A: tool_calls 가 있는 AIMessage → tools 로
print(route([HumanMessage('hi'), ai]))
# 케이스 B: 일반 AIMessage → END
print(route([HumanMessage('hi'), AIMessage('일반 답변')]))

# 도구 결과만 골라내기 (ToolMessage 후처리 패턴)
tool_results = [m.content for m in out["messages"] if isinstance(m, ToolMessage)]
print("도구 결과들:", tool_results)

## 4. 프롬프트를 역할별로 *조립*한다

LLM 에 보낼 프롬프트도 역할 단위로 짠다. `(role, content)` 튜플 리스트나 메시지 객체 리스트로 구성한다. system 으로 페르소나/지시를 주고, human 으로 실제 입력을 준다.

> 아래는 LLM 호출 없이 '프롬프트가 어떤 메시지들로 구성되는지' 만 확인한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 영어를 한국어로 번역하는 도우미야."),
    ("user", "Translate: {text}"),
])

# 템플릿을 채우면 실제 메시지 객체 리스트가 만들어진다
built = prompt.format_messages(text="Hello, world!")
for m in built:
    print(f"{type(m).__name__:14s} | {m.content!r}")

## 정리

- 대화 = **역할(role)이 붙은 메시지 리스트**. `System`/`Human`/`AI`/`Tool` 네 타입이 핵심
- 노드는 타입을 **생성**한다: chatbot→`AIMessage`(+tool_calls), tools(`ToolNode`)→`ToolMessage`
- 라우터/노드는 타입·`tool_calls` 유무로 **분기**한다 (그래프 제어의 핵심)
- 프롬프트는 역할별(`system`/`user`)로 **조립**한다
- `tool_call_id` 가 `AIMessage` 의 도구 호출과 `ToolMessage` 결과를 짝지어준다

이 타입 구분은 `projects/02`(Tool Calling), `projects/04`(Structured Output), `projects/05`(영어 회화) 같은 실제 에이전트에서 라우팅·후처리의 기준으로 쓰인다.